# Engenharia de Features e Pré-processamento Blindado contra Vazamento de Dados

Este notebook corresponde à **Etapa 5** do nosso Roteiro de Desenvolvimento do **Tech Challenge (Fase 1)**. 

### 🎯 Objetivo desta Etapa:
O principal foco do pré-processamento em Ciência de Dados é a **blindagem contra o Vazamento de Dados (*Data Leakage*)**. O vazamento ocorre quando informações do conjunto de teste (ou dados futuros) "vazam" para o conjunto de treinamento através de cálculos de estatísticas globais (como a média e desvio padrão para normalização ou preenchimento de nulos).

Para evitar esse erro clássico, utilizaremos a biblioteca **Scikit-Learn** para construir pipelines modulares de transformação:
1. **`ColumnTransformer`**: Permite aplicar transformações independentes e paralelas em diferentes tipos de colunas (numéricas vs. categóricas).
2. **`StandardScaler`**: Padronização de variáveis numéricas de forma isolada.
3. **`OneHotEncoder`**: Codificação robusta de variáveis categóricas com tratamento para categorias desconhecidas (`handle_unknown='ignore'`).
4. **`SimpleImputer`**: Imputação segura para evitar falhas com eventuais dados faltantes.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Configuração de logs simples
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

## 📂 1. Carregando as Configurações e Dados Processados

Vamos importar as variáveis de configuração que definimos na **Etapa 2** (`src/config.py`) para garantir que as exclusões de colunas sensíveis e futuras ocorram exatamente como planejado.

In [ ]:
# Ajuste do caminho relativo para rodar a partir do repositório local
BASE_DIR = Path(os.getcwd()).resolve()
if BASE_DIR.name == 'notebooks':
    BASE_DIR = BASE_DIR.parent

PROCESSED_DATA_PATH = BASE_DIR / "data" / "processed" / "processed_nps_data.csv"

# Fallback caso esteja rodando num ambiente temporário
if not PROCESSED_DATA_PATH.exists():
    PROCESSED_DATA_PATH = Path("processed_nps_data.csv")

print(f"Carregando dados processados de: {PROCESSED_DATA_PATH.resolve()}")
df = pd.read_csv(PROCESSED_DATA_PATH)
print(f"Dataset carregado! Linhas: {df.shape[0]} | Colunas: {df.shape[1]}")
df.head()

## 🔒 2. Divisão de Features (X) e Target (y)

Para evitar qualquer vazamento de dados, excluiremos colunas de IDs (`customer_id`, `order_id`), a nota contínua de origem (`nps_score`), e as informações que acontecem após a entrega ou em momentos incertos (`repeat_purchase_30d` e `csat_internal_score`).

In [ ]:
# Exclusões estritas baseadas no config.py
EXCLUDED_COLS = [
    "customer_id",
    "order_id",
    "nps_score",
    "repeat_purchase_30d",
    "csat_internal_score"
]

# Também removemos colunas de faixas categóricas auxiliares que criamos apenas para a AED do Passo 4
AUX_COLS = ["delay_group", "contacts_group", "support_group"]

drop_cols = EXCLUDED_COLS + AUX_COLS + ["is_detractor"]
available_drops = [col for col in drop_cols if col in df.columns]

# Divisão X e y
X = df.drop(columns=available_drops)
y = df["is_detractor"]

print("=== SHAPES ORIGINAIS ===")
print(f"X (Features): {X.shape}")
print(f"y (Target): {y.shape}")
print(f"Proporção de Detratores: {y.mean() * 100:.2f}%")

## 🛠️ 3. Separação de Tipos de Colunas para Pré-processamento

Precisamos separar as features em numéricas (que serão escalonadas) e categóricas (que serão convertidas em variáveis binárias One-Hot).

In [ ]:
num_features = []
cat_features = []

for col in X.columns:
    if pd.api.types.is_numeric_dtype(X[col]):
        num_features.append(col)
    else:
        cat_features.append(col)

print(f"-> Features Numéricas ({len(num_features)}):\n{num_features}\n")
print(f"-> Features Categóricas ({len(cat_features)}):\n{cat_features}")

## ⚙️ 4. Construindo o ColumnTransformer do Scikit-Learn

Agora criamos o pipeline de pré-processamento. Este pipeline unificado se integrará perfeitamente aos modelos preditivos nas próximas etapas de treinamento.

In [ ]:
# 1. Sub-pipeline Numérico
num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")), # Imputação segura pela mediana
    ("scaler", StandardScaler())                 # Padronização (Z-score)
])

# 2. Sub-pipeline Categórico
cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)) # Evita quebra com dados desconhecidos
])

# 3. Unificador Global (ColumnTransformer)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, num_features),
        ("cat", cat_transformer, cat_features)
    ],
    remainder="drop" # Descarta qualquer outra coluna que passe batida
)

print("✓ ColumnTransformer instanciado com sucesso!")

## 🧪 5. Executando um fit_transform de Validação

Para certificar que nosso pipeline está pronto para entrar em produção, vamos simular a transformação sobre o nosso X e inspecionar a base resultante.

In [ ]:
# Aplicando o pré-processador
X_trans = preprocessor.fit_transform(X)

print("=== COMPARAÇÃO DE DIMENSÕES ===")
print(f"Shape original de X: {X.shape}")
print(f"Shape após transformações: {X_trans.shape}")

# Obtendo o nome das colunas transformadas após o OneHotEncoder para fins visuais
encoded_cats = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(cat_features)
all_feature_names = num_features + list(encoded_cats)

print(f"\n✓ Total de colunas geradas no pipeline: {len(all_feature_names)}")
print("Nomes das colunas pós-transformação:")
for i, col_name in enumerate(all_feature_names):
    print(f" - [{i}]: {col_name}")

## 📝 6. Visualizando a Matriz Resultante

Vamos converter a matriz NumPy transformada em um DataFrame pandas apenas para inspecionar os primeiros registros e comprovar que os dados estão normalizados de forma correta e sem nulos.

In [ ]:
df_trans = pd.DataFrame(X_trans, columns=all_feature_names)
print("Estatísticas resumidas após padronização (Média ~0, Desvio Padrão ~1):")
display(df_trans.describe().round(3).head(3)) # Médias e desvios padronizados

print("\nPrimeiros 3 registros pós-processamento:")
df_trans.head(3)

### 🚀 Próximos Passos:
Com as features estruturadas e protegidas, estamos prontos para a **Etapa 6 (Treinamento, Seleção e Auditoria)**, na qual utilizaremos este pré-processador de forma blindada em uma validação cruzada estratificada de 5 folds para treinar e comparar algoritmos como Regressão Logística e Random Forest!